In [0]:
storage_account = "casestudy4ecommerce"
storage_key = "dXfbvHOmPknoxQTYTesQsGn6w+Kd9tyjzsZiZPtNYEMYKI516SGzKrVUkszG6XXNEEPyjlIzYvbK+AStwjpdOQ=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("Storage authentication configured")

Storage authentication configured


In [0]:
raw_path = "abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/raw/transactions.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_path)
)

display(df)

order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date
1001,C001,P101,Laptop,Electronics,1,55000,Completed,2026-08-30,2026-08-30
1002,C002,P102,Mobile,Electronics,2,30000,Completed,2026-08-30,2026-08-30
1003,C003,P103,Headphones,Accessories,1,3000,pending,2026-08-30,2026-08-30
1004,C001,P104,Keyboard,Accessories,1,1500,completed,2026-08-30,2026-08-30
1005,C004,P105,Mouse,Accessories,2,800,Cancelled,2026-08-30,2026-08-30
1006,C005,P106,Monitor,Electronics,1,12000,Completed,2026-08-30,2026-08-30
1007,C006,P107,Watch,Wearables,1,7000,Completed,2026-08-30,2026-08-30


creating bronze layer

In [0]:
from pyspark.sql.functions import current_timestamp
bronze_df = df.withColumn("ingestion_timestamp", current_timestamp())
display(bronze_df)

order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp
1001,C001,P101,Laptop,Electronics,1,55000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1002,C002,P102,Mobile,Electronics,2,30000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1003,C003,P103,Headphones,Accessories,1,3000,pending,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1004,C001,P104,Keyboard,Accessories,1,1500,completed,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1005,C004,P105,Mouse,Accessories,2,800,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1006,C005,P106,Monitor,Electronics,1,12000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z
1007,C006,P107,Watch,Wearables,1,7000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:21.821Z


In [0]:
bronze_path = "abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/bronze/transactions"
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

print("Bronze Delta table created successfully")

Bronze Delta table created successfully


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)
print("Bronze record count:", bronze_df.count())
display(bronze_df)

Bronze record count: 7


order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp
1001,C001,P101,Laptop,Electronics,1,55000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1002,C002,P102,Mobile,Electronics,2,30000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1003,C003,P103,Headphones,Accessories,1,3000,pending,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1004,C001,P104,Keyboard,Accessories,1,1500,completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1005,C004,P105,Mouse,Accessories,2,800,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1006,C005,P106,Monitor,Electronics,1,12000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1007,C006,P107,Watch,Wearables,1,7000,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z


cleaning data

In [0]:
from pyspark.sql.functions import *

silver_df = (
    bronze_df
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("order_date", to_date("order_date"))
    .withColumn("modified_date", to_date("modified_date"))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("payment_status", initcap(trim(col("payment_status"))))
    .dropDuplicates(["order_id"])
)

display(silver_df)

order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp
1001,C001,P101,Laptop,Electronics,1,55000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1002,C002,P102,Mobile,Electronics,2,30000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1003,C003,P103,Headphones,Accessories,1,3000.0,Pending,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1004,C001,P104,Keyboard,Accessories,1,1500.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1005,C004,P105,Mouse,Accessories,2,800.0,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1006,C005,P106,Monitor,Electronics,1,12000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1007,C006,P107,Watch,Wearables,1,7000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z


data quality checks

In [0]:
dq_df = silver_df.filter(
    (col("order_id").isNotNull()) &
    (col("customer_id").isNotNull()) &
    (col("product_id").isNotNull()) &
    (col("quantity") > 0) &
    (col("price") > 0) &
    (col("order_date").isNotNull()) &
    (col("modified_date").isNotNull())
)

total_records = bronze_df.count()
valid_records = dq_df.count()
rejected_records = total_records - valid_records

print("Total Bronze records:", total_records)
print("Valid Silver records:", valid_records)
print("Rejected records:", rejected_records)

Total Bronze records: 7
Valid Silver records: 7
Rejected records: 0


In [0]:
silver_path = "abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/silver/transactions"

dq_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_path)

print("Silver Delta table created successfully")

Silver Delta table created successfully


In [0]:
silver_df = spark.read.format("delta").load(silver_path)
print("Silver record count:", silver_df.count())
display(silver_df)

Silver record count: 7


order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp
1005,C004,P105,Mouse,Accessories,2,800.0,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1002,C002,P102,Mobile,Electronics,2,30000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1001,C001,P101,Laptop,Electronics,1,55000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1006,C005,P106,Monitor,Electronics,1,12000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1007,C006,P107,Watch,Wearables,1,7000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1003,C003,P103,Headphones,Accessories,1,3000.0,Pending,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z
1004,C001,P104,Keyboard,Accessories,1,1500.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z


gold

In [0]:
gold_df = (
    silver_df
    .withColumn(
        "sales_amount",
        col("quantity") * col("price")
    )
    .withColumn(
        "is_successful_order",
        when(col("payment_status") == "Completed", 1)
        .otherwise(0)
    )
)

display(gold_df)

order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp,sales_amount,is_successful_order
1005,C004,P105,Mouse,Accessories,2,800.0,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,1600.0,0
1002,C002,P102,Mobile,Electronics,2,30000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,60000.0,1
1001,C001,P101,Laptop,Electronics,1,55000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,55000.0,1
1006,C005,P106,Monitor,Electronics,1,12000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,12000.0,1
1007,C006,P107,Watch,Wearables,1,7000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,7000.0,1
1003,C003,P103,Headphones,Accessories,1,3000.0,Pending,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,3000.0,0
1004,C001,P104,Keyboard,Accessories,1,1500.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,1500.0,1


In [0]:
gold_path = "abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/gold/order_sales"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(gold_path)

print("Gold Delta table created successfully")

Gold Delta table created successfully


In [0]:
gold_df = spark.read.format("delta").load(gold_path)
print("Gold record count:", gold_df.count())
display(gold_df)

Gold record count: 7


order_id,customer_id,product_id,product_name,category,quantity,price,payment_status,order_date,modified_date,ingestion_timestamp,sales_amount,is_successful_order
1005,C004,P105,Mouse,Accessories,2,800.0,Cancelled,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,1600.0,0
1002,C002,P102,Mobile,Electronics,2,30000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,60000.0,1
1001,C001,P101,Laptop,Electronics,1,55000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,55000.0,1
1006,C005,P106,Monitor,Electronics,1,12000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,12000.0,1
1007,C006,P107,Watch,Wearables,1,7000.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,7000.0,1
1003,C003,P103,Headphones,Accessories,1,3000.0,Pending,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,3000.0,0
1004,C001,P104,Keyboard,Accessories,1,1500.0,Completed,2026-08-30,2026-08-30,2026-08-30T14:45:59.389Z,1500.0,1


In [0]:
gold_df.createOrReplaceTempView("gold_order_sales")

total completed sales

In [0]:
%sql

SELECT
    SUM(sales_amount) AS total_completed_sales
FROM gold_order_sales
WHERE payment_status = 'Completed';

total_completed_sales
135500.0


sales by category

In [0]:
%sql

SELECT
    category,
    SUM(quantity) AS total_quantity,
    SUM(sales_amount) AS total_sales
FROM gold_order_sales
WHERE payment_status = 'Completed'
GROUP BY category
ORDER BY total_sales DESC;

category,total_quantity,total_sales
Electronics,4,127000.0
Wearables,1,7000.0
Accessories,1,1500.0


customer analysis

In [0]:
%sql

SELECT
    customer_id,
    COUNT(order_id) AS total_orders,
    SUM(sales_amount) AS total_sales
FROM gold_order_sales
WHERE payment_status = 'Completed'
GROUP BY customer_id
ORDER BY total_sales DESC;

customer_id,total_orders,total_sales
C002,1,60000.0
C001,2,56500.0
C005,1,12000.0
C006,1,7000.0


In [0]:
from delta.tables import DeltaTable
gold_delta = DeltaTable.forPath(spark, gold_path)

In [0]:
incremental_gold = (
    silver_df
    .withColumn(
        "sales_amount",
        col("quantity") * col("price")
    )
    .withColumn(
        "is_successful_order",
        when(col("payment_status") == "Completed", 1)
        .otherwise(0)
    )
)

In [0]:
gold_delta.alias("target") \
    .merge(
        incremental_gold.alias("source"),
        "target.order_id = source.order_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("Incremental MERGE completed successfully")

Incremental MERGE completed successfully


In [0]:
%sql

OPTIMIZE delta.`abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/gold/order_sales`;

path,metrics
abfss://datacontainer@casestudy4ecommerce.dfs.core.windows.net/gold/order_sales,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, 0, 1, 1, true, 0, 0, 1788101716948, 1788101717961, 4, 0, null, List(0, 0), 13, 13, 0, 0, null)"
